In [ ]:
# pip install langchain langchain-openai requests

In [7]:
import langchain
import requests
import importlib.metadata

print(f"LangChain version: {langchain.__version__}")
print(f"Requests version: {requests.__version__}")

try:
    l_openai_version = importlib.metadata.version("langchain-openai")
    print(f"LangChain-OpenAI version: {l_openai_version}")
except importlib.metadata.PackageNotFoundError:
    print("LangChain-OpenAI is not installed.")


LangChain version: 1.2.15
Requests version: 2.33.1
LangChain-OpenAI version: 1.2.1


In [8]:
# Ideally put this in your enviroment

OPENAI_API_KEY = "key here"

In [10]:
import requests
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent


# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0
)

# TOOLS
@tool
def calculator(expression: str) -> str:
    """Perform mathematical calculations."""

    print("...Running calculator tool")
    try: 
        return str(eval(expression))
    except: 
        return "Error in calculation"

@tool
def get_weather(city: str) -> str:
    """Get current weather of a city."""
    
    print("...Running weather tool")
    try:
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
        geo_res = requests.get(geo_url).json()

        if "results" not in geo_res:
            return "City not found"

        lat = geo_res["results"][0]["latitude"]
        lon = geo_res["results"][0]["longitude"]

        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_res = requests.get(weather_url).json()

        temp = weather_res["current_weather"]["temperature"]
        wind = weather_res["current_weather"]["windspeed"]

        return f"Temperature: {temp}°C, Wind Speed: {wind} km/h"

    except:
        return "Error fetching weather"


In [11]:
tools = [calculator, get_weather]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant with access to tools."
)

## Try
```
input1: What is 1 + 2 * 4 ?

input2: What is the current weather condition in Moscow ?

input3: I am trying to calculate this math expression. What is one plus two times four ?

input4: I am living in Moscow. I was thinking of going to the park. I wonder how is weather outside. Is it going going to be nice today ?

In [12]:
user_input = "What is 1 + 2 * 4 ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running calculator tool
Response: {'messages': [HumanMessage(content='What is 1 + 2 * 4 ?', additional_kwargs={}, response_metadata={}, id='9d36b0f1-4280-47f2-b162-8fecd496f544'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 82, 'total_tokens': 102, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b6580bbee1', 'id': 'chatcmpl-DZrwhPh8V3sgzcPyq7m41ZpEJdAe2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd7c0-99ee-7392-86b2-66bd90c2fef6-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '1 + 2 * 4'}, 'id': 'call_UxLBPJd0Q1nXVMRLYosaBWZQ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_me

In [13]:
user_input = "What is the current weather condition in Moscow ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running weather tool
Response: {'messages': [HumanMessage(content='What is the current weather condition in Moscow ?', additional_kwargs={}, response_metadata={}, id='bf939dc2-c393-4293-b390-e89304537b28'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 80, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ed279b101f', 'id': 'chatcmpl-DZrwzcPslQcVAMvfCluQtNjfxlFL0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd7c0-e36c-7a61-b530-94456ae93ecc-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Moscow'}, 'id': 'call_PCTmqlp4dRP7BIg1YMSH7Qgd', 'type': 'tool_call'}], invalid_tool_

In [14]:
user_input = "I am trying to calculate this math expression. What is one plus two times four ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running calculator tool
Response: {'messages': [HumanMessage(content='I am trying to calculate this math expression. What is one plus two times four ?', additional_kwargs={}, response_metadata={}, id='c0cf7717-1afd-474a-948d-c7c3bc267ccf'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 88, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b6580bbee1', 'id': 'chatcmpl-DZrxJugJ0wi8qD1C4Xg8f1OmG02lu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd7c1-3431-7e90-86a6-2ba4865a1422-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '1 + 2 * 4'}, 'id': 'call_siWkPODMy9WKFx38iZrb

In [15]:
user_input = "I am living in Moscow. I was thinking of going to the park. \
              I wonder how is weather outside. Is it going going to be nice today ?"

response = agent.invoke(
    {"messages": [{"role": "user", 
                   "content": user_input
                  }]
    }
)

print("Response:", response)
print("Final Answer:", response["messages"][-1].content)

...Running weather tool
Response: {'messages': [HumanMessage(content='I am living in Moscow. I was thinking of going to the park.               I wonder how is weather outside. Is it going going to be nice today ?', additional_kwargs={}, response_metadata={}, id='0471f36b-1e52-4012-99a7-c61ca30c0fd4'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 103, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b6580bbee1', 'id': 'chatcmpl-DZrxSpzFPZ4kTOyEIJJdvLCFGAupC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd7c1-5610-7f43-a9eb-ffb9bc78b566-0', tool_calls=[{'name': 'get_weather', 'args':